# Module 06-1: Pydantic 모델로 출력 스키마 정의 (솔루션)

## 학습 목표
- `BaseModel`과 `Field`로 데이터 스키마를 정의할 수 있다
- `Field` 제약조건(ge, le, pattern, min_length)을 사용하여 데이터를 검증할 수 있다
- `Enum`으로 허용값을 제한할 수 있다
- `field_validator`로 커스텀 검증 로직을 추가할 수 있다

**참조 문서**: `docs/part3-prompt-and-llm/06-structured-output.md` 섹션 2.1

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from pydantic import BaseModel, Field, field_validator, ValidationError
from enum import Enum
from typing import Optional

print("환경 설정 완료")

## 실습 1: 기본 BaseModel과 Field 제약조건 (솔루션)

In [ ]:
# 솔루션
class BugReport(BaseModel):
    """버그 리포트 스키마."""
    line: int = Field(description="버그가 있는 라인 번호", ge=1)
    issue: str = Field(description="버그 설명", min_length=1)
    severity: str = Field(description="심각도", pattern=r"^(HIGH|MEDIUM|LOW)$")


valid_bug = BugReport(line=5, issue="null check missing", severity="HIGH")
print("올바른 데이터:", valid_bug.model_dump())

try:
    bad_bug = BugReport(line=-1, issue="", severity="CRITICAL")
except ValidationError as e:
    print(f"ValidationError 발생: {len(e.errors())}개 오류")

In [ ]:
# 검증 셀
test_bug = BugReport(line=1, issue="test issue", severity="MEDIUM")
assert test_bug.line == 1
assert test_bug.issue == "test issue"
assert test_bug.severity == "MEDIUM"

try:
    BugReport(line=0, issue="test", severity="INVALID")
    assert False
except ValidationError:
    pass

try:
    BugReport(line=1, issue="", severity="HIGH")
    assert False
except ValidationError:
    pass

print("실습 1 완료!")

## 실습 2: Enum으로 허용값 제한 (솔루션)

In [ ]:
# 솔루션
class Severity(str, Enum):
    HIGH = "HIGH"
    MEDIUM = "MEDIUM"
    LOW = "LOW"


class TestStatus(str, Enum):
    PASS = "PASS"
    FAIL = "FAIL"
    BLOCKED = "BLOCKED"


class TestResult(BaseModel):
    """테스트 결과 스키마."""
    status: TestStatus
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str = Field(min_length=1)
    severity: Severity = Field(default=Severity.MEDIUM)


result = TestResult(status=TestStatus.PASS, confidence=0.95, reasoning="모든 테스트 케이스 통과")
print("TestResult 생성:", result.model_dump())
print(f"severity 기본값: {result.severity}")

In [ ]:
# 검증 셀
assert hasattr(Severity, 'HIGH')
assert hasattr(Severity, 'MEDIUM')
assert hasattr(Severity, 'LOW')
assert hasattr(TestStatus, 'PASS')
assert hasattr(TestStatus, 'FAIL')
assert hasattr(TestStatus, 'BLOCKED')

test_result = TestResult(status="PASS", confidence=0.8, reasoning="테스트 통과")
assert test_result.status == TestStatus.PASS

try:
    TestResult(status="PASS", confidence=1.5, reasoning="테스트")
    assert False
except ValidationError:
    pass

print("실습 2 완료!")

## 실습 3: field_validator로 커스텀 검증 (솔루션)

In [ ]:
# 솔루션
class CodeAnalysis(BaseModel):
    """코드 분석 결과 스키마."""
    affected_files: list[str] = Field(min_length=1, description="영향받는 파일 목록")
    confidence: float = Field(ge=0.0, le=1.0, description="분석 신뢰도")
    root_cause: str = Field(min_length=1, description="근본 원인")

    @field_validator("confidence")
    @classmethod
    def round_confidence(cls, v: float) -> float:
        return round(v, 2)

    @field_validator("affected_files")
    @classmethod
    def validate_file_paths(cls, v: list[str]) -> list[str]:
        return [f for f in v if f.strip()]


analysis = CodeAnalysis(
    affected_files=["src/main.py", "", "src/utils.py"],
    confidence=0.85678,
    root_cause="Missing null check"
)
print(f"confidence: {analysis.confidence} (0.85678 -> 소수점 2자리)")
print(f"affected_files: {analysis.affected_files} (빈 문자열 제거됨)")

In [ ]:
# 검증 셀
test_analysis = CodeAnalysis(
    affected_files=["main.py", "", "utils.py"],
    confidence=0.12345,
    root_cause="테스트 원인"
)
assert test_analysis.confidence == 0.12, f"confidence가 0.12여야 합니다. 현재: {test_analysis.confidence}"
assert "" not in test_analysis.affected_files
assert len(test_analysis.affected_files) == 2
print("실습 3 완료!")

## 실습 4: 중첩 모델 (완성된 예시)

In [ ]:
# 중첩 모델 정의
class SuggestedFix(BaseModel):
    description: str = Field(description="수정 방법 설명")
    code_snippet: str = Field(default="", description="수정 코드 예시")

class BugAnalysis(BaseModel):
    title: str = Field(min_length=1, description="버그 제목")
    severity: Severity = Field(description="심각도")
    description: str = Field(min_length=10, description="상세 설명")
    suggested_fix: SuggestedFix = Field(description="수정 제안")

class BugReportAnalysis(BaseModel):
    total_bugs: int = Field(ge=0, description="발견된 총 버그 수")
    bugs: list[BugAnalysis] = Field(default_factory=list)
    summary: str = Field(min_length=1, description="분석 요약")


report = BugReportAnalysis(
    total_bugs=1,
    bugs=[BugAnalysis(
        title="SQL Injection 취약점",
        severity=Severity.HIGH,
        description="사용자 입력을 직접 SQL 쿼리에 삽입하여 SQL Injection 공격에 취약합니다.",
        suggested_fix=SuggestedFix(description="파라미터화된 쿼리 사용", code_snippet="db.execute('SELECT * FROM users WHERE id = ?', (user_id,))")
    )],
    summary="1개의 심각한 보안 취약점 발견"
)
print(f"총 버그 수: {report.total_bugs}")
print(f"첫 번째 버그: {report.bugs[0].title}")

In [ ]:
# 검증 셀
assert report.total_bugs == 1
assert len(report.bugs) == 1
assert isinstance(report.bugs[0], BugAnalysis)
assert isinstance(report.bugs[0].suggested_fix, SuggestedFix)
print("실습 4 완료!")

## 정리

1. **BaseModel + Field**: 스키마 정의와 제약조건 설정
2. **Field 제약조건**: `ge`, `le`, `min_length`, `max_length`, `pattern`, `default`, `default_factory`
3. **Enum**: 허용값을 명시적으로 제한하여 타입 안전성 확보
4. **field_validator**: 커스텀 검증 로직 (값 변환, 필터링)
5. **중첩 모델**: 복잡한 데이터 구조를 계층적으로 표현